## imports

In [1]:
import pandas as pd

The project CSV files are stored in the data/raw

In [2]:
ad_revenue = pd.read_csv('../data/raw/ad_revenue.csv')
iap_products = pd.read_csv('../data/raw/iap_products.csv')
matches = pd.read_csv('../data/raw/matches.csv')
purchases = pd.read_csv('../data/raw/purchases.csv')
sessions = pd.read_csv('../data/raw/sessions.csv')
ua_spend = pd.read_csv('../data/raw/ua_spend.csv')
users = pd.read_csv('../data/raw/users.csv')

In [3]:
# Date parsing helper for mixed formats
def parse_mixed_date(series):
    s = series.astype(str).str.strip()
    parsed = pd.Series(index=s.index, dtype='datetime64[ns]')
    iso_mask = s.str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False)
    slash_mask = s.str.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}', na=False)
    text_mask = s.str.lower().str.fullmatch(r'\d{1,2}\s+[a-záéíóúñ]{3}\s+\d{4}', na=False)
    month_map = {'ene': '01', 'feb': '02', 'mar': '03', 'abr': '04', 'may': '05', 'jun': '06', 'jul': '07', 'ago': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dic': '12'}
    parsed.loc[iso_mask] = pd.to_datetime(s.loc[iso_mask], format='%Y-%m-%d', errors='coerce')
    parsed.loc[slash_mask] = pd.to_datetime(s.loc[slash_mask], format='%d/%m/%Y', errors='coerce')
    if text_mask.any():
        text_parts = s.loc[text_mask].str.lower().str.replace('.', '', regex=False).str.extract(r'(?P<day>\d{1,2})\s+(?P<mon>[a-záéíóúñ]{3})\s+(?P<year>\d{4})')
        normalized_text = text_parts['day'].str.zfill(2) + '/' + text_parts['mon'].map(month_map) + '/' + text_parts['year']
        parsed.loc[text_mask] = pd.to_datetime(normalized_text, format='%d/%m/%Y', errors='coerce')
    remaining = ~(iso_mask | slash_mask | text_mask)
    if remaining.any():
        parsed.loc[remaining] = pd.to_datetime(s.loc[remaining], dayfirst=True, errors='coerce')
    return parsed


file: ad_revenue

In [4]:
# Date Format
ad_revenue['fecha'] = parse_mixed_date(ad_revenue['fecha'])

#text normalize
map_type = {'recompensado':'rewarded'}
ad_revenue['tipo_ad'] = ad_revenue['tipo_ad'].str.strip().str.lower().replace(map_type)

ad_revenue['red_ad'] = ad_revenue['red_ad'].str.strip().str.lower()

file: iap_products

In [5]:
# format
iap_products['precio_base'] = iap_products['precio_base'].str.strip().str.upper().str.replace('€','', regex=False).str.replace(',','.', regex=False).astype(float)


#text normalize
iap_products['tipo'] = iap_products['tipo'].str.strip().str.lower()




file: matches

In [6]:
# Date Format
matches['fecha'] = parse_mixed_date(matches['fecha'])

# Text normalize
map_resultado = {'fail': 'lose', 'ganada':'win', 'perdida':'lose'}
matches['resultado'] = matches['resultado'].str.strip().str.lower().replace(map_resultado)

file: purchases

In [7]:
# Date Format
purchases['fecha'] = parse_mixed_date(purchases['fecha'])

def norm_plat(x):
    s = str(x).strip().lower()
    if s == 'ios':
        return 'ios'
    if s == 'and':
        return 'android'
    return s

# clearing of securities
purchases['moneda'] =purchases['moneda'].astype(str).str.strip().str.upper().replace({'€': 'EUR'})
purchases['plataforma'] = purchases['plataforma'].str.strip().str.lower().apply(norm_plat)
purchases['importe'] = purchases['importe'].str.replace('€', '', regex=False).str.replace('MXN', '', regex=False).str.replace(',', '.', regex=False).str.strip().astype(float)

sales = purchases[(purchases['reembolso'] == 0)&(purchases['moneda'] == 'EUR')].copy()

print(f"Ventas EUR validas: {len(sales)}")
print(f"Ingreso IAP bruto: {sales['importe'].sum():.2f} EUR")

Ventas EUR validas: 4479
Ingreso IAP bruto: 10838.71 EUR


In [19]:
purchases['importe'].describe()

count    4558.000000
mean        2.684697
std        14.934021
min       -19.990000
25%         0.990000
50%         1.990000
75%         2.490000
max       974.810000
Name: importe, dtype: float64

file: sessions

In [9]:
# Date Format
sessions['fecha'] = parse_mixed_date(sessions['fecha'])


file: ua_spend

In [10]:
# Date Format
ua_spend['fecha'] = parse_mixed_date(ua_spend['fecha'])

file: users

In [11]:
# Date Format
users['fecha_instalacion'] = parse_mixed_date(users['fecha_instalacion'])
